# Megaminx Q Model Inference Standalone

This notebook runs inference for the Megaminx Q model without importing the local `pilgrim` package. It only needs the model artifacts next to it: generator spec, target tensor, metadata JSON, and checkpoint.

Default artifact set is the newer local symmetric Q checkpoint:

- `generators/p900.json`
- `targets/p900-t000.pt`
- `logs/model_p900-t000-q-sym_1777988767.json`
- `weights/p900-t000-q-sym_1777988767_best.pth`

Put this notebook in the repository `notebooks/` directory, or put those four folders next to the notebook and run all cells top to bottom.

In [ ]:
# If torch is missing, install a build matching your machine first:
# CPU example:      pip install torch tqdm
# CUDA example:     install torch from https://pytorch.org/get-started/locally/, then pip install tqdm

from pathlib import Path
import json
import math
import time
from collections import deque

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())

In [ ]:
CONFIG = {
    'group_id': 900,
    'target_id': 0,
    'model_name': 'p900-t000-q-sym',
    'model_id': 1777988767,
    'checkpoint': 'best',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'eval_batch_size': 2**14,
    'search_seed': 0,
    # Smoke-search defaults. Increase B/num_steps for harder states.
    'rnd_depth': 20,
    'rnd_seed': 42,
    'beam_width': 4096,
    'num_steps': 80,
    'num_attempts': 1,
}

# Search roots cover both layouts:
# 1. running from cayleypy-neighbour-model-training-public/notebooks
# 2. running from a standalone folder with generators/logs/targets/weights beside the notebook
ASSET_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'cayleypy-neighbour-model-training-public']


def find_asset(relative_path):
    relative_path = Path(relative_path)
    tried = []
    for root in ASSET_ROOT_CANDIDATES:
        candidate = (root / relative_path).resolve()
        tried.append(str(candidate))
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find ' + str(relative_path) + '\nTried:\n' + '\n'.join(tried))


def artifact_paths(cfg):
    gid = int(cfg['group_id'])
    tid = int(cfg['target_id'])
    model_name = cfg['model_name']
    model_id = int(cfg['model_id'])
    checkpoint = cfg['checkpoint']
    if checkpoint == 'best':
        weight_name = f'{model_name}_{model_id}_best.pth'
    else:
        weight_name = f'{model_name}_{model_id}_e{int(checkpoint):05d}.pth'
    return {
        'generator': find_asset(f'generators/p{gid:03d}.json'),
        'target': find_asset(f'targets/p{gid:03d}-t{tid:03d}.pt'),
        'metadata': find_asset(f'logs/model_{model_name}_{model_id}.json'),
        'weights': find_asset(f'weights/{weight_name}'),
    }

paths = artifact_paths(CONFIG)
paths

In [ ]:
def torch_load(path, *, map_location=None, weights_only=None):
    kwargs = {}
    if map_location is not None:
        kwargs['map_location'] = map_location
    if weights_only is not None:
        try:
            return torch.load(path, weights_only=weights_only, **kwargs)
        except TypeError:
            return torch.load(path, **kwargs)
    return torch.load(path, **kwargs)


def parse_generator_spec(data):
    if 'moves' in data and 'move_names' in data:
        return data['moves'], data['move_names']
    if 'actions' in data and 'names' in data:
        return data['actions'], data['names']
    raise KeyError('generator spec must contain either moves/move_names or actions/names')


def generate_inverse_moves(move_names):
    inverse_moves = [0] * len(move_names)
    for i, move in enumerate(move_names):
        inverse_moves[i] = move_names.index(move.replace("'", '')) if "'" in move else move_names.index(move + "'")
    return inverse_moves


def state2hash(states, hash_vec, batch_size=2**14):
    result = torch.empty(states.size(0), dtype=torch.int64, device=states.device)
    for start in range(0, states.size(0), batch_size):
        batch = states[start:start + batch_size].to(torch.int64)
        result[start:start + batch_size] = torch.sum(hash_vec * batch, dim=1)
    return result


def generate_random_walk_states(V0, all_moves, inverse_moves, num_states, depth, device, seed=0):
    if num_states <= 0:
        raise ValueError('num_states must be > 0')
    if depth < 0:
        raise ValueError('depth must be >= 0')
    states = V0.repeat(num_states, 1)
    if depth == 0:
        return states
    inverse_moves_cpu = torch.as_tensor(inverse_moves, dtype=torch.int64, device='cpu')
    previous_moves = torch.full((num_states,), -1, dtype=torch.int64)
    rng = torch.Generator(device='cpu')
    rng.manual_seed(int(seed))
    for _ in range(depth):
        next_moves = torch.randint(all_moves.size(0), (num_states,), generator=rng, dtype=torch.int64)
        invalid = (previous_moves >= 0) & (next_moves == inverse_moves_cpu[previous_moves.clamp_min(0)])
        while invalid.any():
            next_moves[invalid] = torch.randint(all_moves.size(0), (int(invalid.sum().item()),), generator=rng, dtype=torch.int64)
            invalid = (previous_moves >= 0) & (next_moves == inverse_moves_cpu[previous_moves.clamp_min(0)])
        states = torch.gather(states, 1, all_moves[next_moves.to(device)])
        previous_moves = next_moves
    return states

In [ ]:
class LegacyCompatibleEmbeddingBagLinear(nn.Module):
    def __init__(self, state_size, num_classes, out_features, bias=True):
        super().__init__()
        self.state_size = int(state_size)
        self.num_classes = int(num_classes)
        self.out_features = int(out_features)
        self.in_features = self.state_size * self.num_classes
        self.weight = nn.Parameter(torch.empty(self.in_features, self.out_features))
        self.bias = nn.Parameter(torch.empty(self.out_features)) if bias else None
        self.register_buffer('position_offsets', torch.arange(self.state_size, dtype=torch.int64) * self.num_classes, persistent=False)
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1.0 / math.sqrt(self.in_features)
        nn.init.uniform_(self.weight, -bound, bound)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, indices):
        if indices.ndim != 2 or indices.size(1) != self.state_size:
            raise ValueError(f'expected indices with shape [batch, {self.state_size}], got {tuple(indices.shape)}')
        token_ids = indices + self.position_offsets.unsqueeze(0)
        out = F.embedding_bag(token_ids, self.weight, mode='sum')
        return out if self.bias is None else out + self.bias

    def _save_to_state_dict(self, destination, prefix, keep_vars):
        destination[prefix + 'weight'] = self.weight.transpose(0, 1) if keep_vars else self.weight.transpose(0, 1).detach()
        if self.bias is not None:
            destination[prefix + 'bias'] = self.bias if keep_vars else self.bias.detach()

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs):
        key = prefix + 'weight'
        if key in state_dict:
            loaded = state_dict[key]
            legacy_shape = (self.out_features, self.in_features)
            native_shape = (self.in_features, self.out_features)
            if loaded.shape == legacy_shape:
                state_dict[key] = loaded.transpose(0, 1)
            elif loaded.shape != native_shape:
                error_msgs.append(f'size mismatch for {key}: expected {legacy_shape} or {native_shape}, got {tuple(loaded.shape)}')
        super()._load_from_state_dict(state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs)


class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

    def forward(self, x):
        residual = x
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.bn2(x)
        return self.relu(x + residual)


class Pilgrim(nn.Module):
    def __init__(self, state_size, hd1=5000, hd2=1000, nrd=2, output_dim=1, dropout_rate=0.0, num_classes=6):
        super().__init__()
        self.dtype = torch.float32
        self.state_size = int(state_size)
        self.num_classes = int(num_classes)
        self.output_dim = int(output_dim)
        self.z_add = 0
        self.input_layer = LegacyCompatibleEmbeddingBagLinear(state_size, num_classes, hd1)
        self.bn1 = nn.BatchNorm1d(hd1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        if hd2 > 0:
            self.hidden_layer = nn.Linear(hd1, hd2)
            self.bn2 = nn.BatchNorm1d(hd2)
            out_hidden = hd2
        else:
            self.hidden_layer = None
            self.bn2 = None
            out_hidden = hd1
        self.residual_blocks = nn.ModuleList([ResidualBlock(hd2, dropout_rate) for _ in range(nrd)]) if nrd > 0 and hd2 > 0 else None
        self.output_layer = nn.Linear(out_hidden, self.output_dim)

    def forward(self, z):
        x = self.input_layer(z.long() + self.z_add).to(self.dtype)
        x = self.dropout(self.relu(self.bn1(x)))
        if self.hidden_layer is not None:
            x = self.dropout(self.relu(self.bn2(self.hidden_layer(x))))
        if self.residual_blocks is not None:
            for block in self.residual_blocks:
                x = block(x)
        x = self.output_layer(x)
        return x.squeeze(-1) if self.output_dim == 1 else x


def build_model_from_info(info, *, num_classes, state_size, output_dim):
    return Pilgrim(
        num_classes=num_classes,
        state_size=state_size,
        output_dim=output_dim,
        dropout_rate=float(info.get('dropout', 0.0)),
        hd1=int(info.get('hd1', 1024)),
        hd2=int(info.get('hd2', 256)),
        nrd=int(info.get('nrd', 4)),
    )


def batch_process(model, data, device, batch_size):
    output_dtype = getattr(model, 'dtype', torch.float32)
    output_dim = int(getattr(model, 'output_dim', 1))
    outputs = torch.empty((data.size(0), output_dim), dtype=output_dtype, device=device) if output_dim != 1 else torch.empty(data.size(0), dtype=output_dtype, device=device)
    with torch.inference_mode():
        for start in range(0, data.size(0), batch_size):
            batch = data[start:start + batch_size]
            out = model(batch)
            if output_dim == 1:
                outputs[start:start + batch_size] = out.view(-1)
            else:
                outputs[start:start + batch_size] = out.view(batch.size(0), output_dim)
    return outputs

In [ ]:
class QSearcher:
    def __init__(self, model, all_moves, V0, device, move_names, inverse_moves, batch_size=2**14, hash_seed=0, normalize_path=True):
        self.device = torch.device(device)
        self.model = model.to(self.device).eval()
        self.all_moves = all_moves.to(self.device)
        self.V0 = V0.to(self.device)
        self.move_names = list(move_names)
        self.inverse_moves = torch.as_tensor(inverse_moves, dtype=torch.int64, device=self.device)
        self.batch_size = int(batch_size)
        self.n_gens = self.all_moves.size(0)
        self.state_size = self.all_moves.size(1)
        self.normalize_path = bool(normalize_path)
        self.move_to_idx = {name: idx for idx, name in enumerate(self.move_names)}
        rng = torch.Generator(device='cpu')
        rng.manual_seed(int(hash_seed))
        self.hash_vec = torch.randint(0, int(1e15), (self.state_size,), generator=rng, dtype=torch.int64).to(self.device)
        self.V0_hash = state2hash(self.V0.unsqueeze(0), self.hash_vec, 1)
        self.inverse_all_moves = torch.empty_like(self.all_moves)
        rows = torch.arange(self.n_gens, device=self.device).unsqueeze(1).expand(-1, self.state_size)
        cols = torch.arange(self.state_size, device=self.device).unsqueeze(0).expand(self.n_gens, -1)
        self.inverse_all_moves[rows, self.all_moves] = cols
        self.hash_after_move_weights = self.hash_vec.index_select(0, self.inverse_all_moves.reshape(-1)).view(self.n_gens, self.state_size)
        solved_batch = self.V0.unsqueeze(0).expand(self.n_gens, -1)
        self.solved_predecessors = torch.gather(solved_batch, 1, self.all_moves[self.inverse_moves])
        self.solved_predecessor_hashes = state2hash(self.solved_predecessors, self.hash_vec, self.n_gens)

    @staticmethod
    def _sorted_isin(values, sorted_reference):
        if sorted_reference.numel() == 0:
            return torch.zeros(values.size(0), dtype=torch.bool, device=values.device)
        pos = torch.searchsorted(sorted_reference, values)
        valid = pos < sorted_reference.numel()
        if not valid.any():
            return valid
        check_pos = pos.clamp_max(sorted_reference.numel() - 1)
        return valid & (sorted_reference.index_select(0, check_pos) == values)

    def pred_q(self, states):
        pred = batch_process(self.model, states.to(self.device), self.device, self.batch_size)
        if pred.ndim != 2 or pred.size(1) != self.n_gens:
            raise ValueError(f'expected Q output [batch, {self.n_gens}], got {tuple(pred.shape)}')
        return pred

    def apply_move(self, states, moves):
        moved = torch.empty(states.size(0), self.state_size, device=self.device, dtype=states.dtype)
        for start in range(0, states.size(0), self.batch_size):
            moved[start:start + self.batch_size] = torch.gather(states[start:start + self.batch_size], 1, self.all_moves[moves[start:start + self.batch_size]])
        return moved

    def hash_after_move(self, states, moves):
        hashed = torch.empty(states.size(0), dtype=torch.int64, device=self.device)
        for start in range(0, states.size(0), self.batch_size):
            batch_states = states[start:start + self.batch_size].to(torch.int64)
            batch_moves = moves[start:start + self.batch_size]
            batch_weights = self.hash_after_move_weights.index_select(0, batch_moves)
            hashed[start:start + self.batch_size] = torch.sum(batch_states * batch_weights, dim=1)
        return hashed

    def _valid_action_mask(self, states, last_moves):
        mask = torch.ones((states.size(0), self.n_gens), dtype=torch.bool, device=self.device)
        if last_moves is not None:
            valid_last = last_moves >= 0
            if valid_last.any():
                rows = torch.nonzero(valid_last, as_tuple=False).view(-1)
                mask[rows, self.inverse_moves[last_moves.index_select(0, rows)]] = False
        return mask

    def _find_solved_child(self, states, valid_mask, q_values):
        parent_hashes = state2hash(states, self.hash_vec, self.batch_size)
        solved_mask = (parent_hashes.unsqueeze(1) == self.solved_predecessor_hashes.unsqueeze(0)) & valid_mask
        if not solved_mask.any():
            return None
        flat = torch.nonzero(solved_mask.view(-1), as_tuple=False).view(-1)
        parents = torch.div(flat, self.n_gens, rounding_mode='floor')
        moves = torch.remainder(flat, self.n_gens)
        exact = (states.index_select(0, parents) == self.solved_predecessors.index_select(0, moves)).all(dim=1)
        if not exact.any():
            return None
        solved_flat = flat[torch.nonzero(exact, as_tuple=False).view(-1)[0]]
        solved_parent = torch.div(solved_flat, self.n_gens, rounding_mode='floor').view(1)
        solved_move = torch.remainder(solved_flat, self.n_gens).view(1)
        return self.V0.unsqueeze(0), self.V0_hash, q_values[solved_parent, solved_move].view(1), solved_move, solved_parent

    def _candidate_hashes(self, states, flat_indices):
        hashes = torch.empty(flat_indices.size(0), dtype=torch.int64, device=self.device)
        for start in range(0, flat_indices.size(0), self.batch_size):
            batch_flat = flat_indices[start:start + self.batch_size]
            parents = torch.div(batch_flat, self.n_gens, rounding_mode='floor')
            moves = torch.remainder(batch_flat, self.n_gens)
            hashes[start:start + self.batch_size] = self.hash_after_move(states.index_select(0, parents), moves)
        return hashes

    def _first_unique_positions(self, hashes):
        positions = torch.arange(hashes.size(0), dtype=torch.int64, device=self.device)
        unique_hashes, inverse = torch.unique(hashes, return_inverse=True)
        first_pos = torch.full((unique_hashes.size(0),), hashes.size(0), dtype=torch.int64, device=self.device)
        first_pos.scatter_reduce_(0, inverse, positions, reduce='amin', include_self=True)
        return positions[first_pos.index_select(0, inverse) == positions]

    def do_q_step(self, states, visited_hashes, last_moves, B):
        q_values = self.pred_q(states)
        valid_mask = self._valid_action_mask(states, last_moves)
        solved = self._find_solved_child(states, valid_mask, q_values)
        if solved is not None:
            return solved
        q_flat = q_values.masked_fill(~valid_mask, float('inf')).reshape(-1)
        total_valid = int(valid_mask.sum().item())
        if total_valid == 0:
            empty = torch.empty((0,), dtype=torch.int64, device=self.device)
            return torch.empty((0, self.state_size), dtype=states.dtype, device=self.device), empty, empty.float(), empty, empty
        current_k = min(total_valid, max(int(B) * 2, 16384))
        while True:
            _, candidate_indices = torch.topk(q_flat, k=current_k, largest=False, sorted=True)
            candidate_hashes = self._candidate_hashes(states, candidate_indices)
            if visited_hashes.numel() > 0:
                keep_mask = ~self._sorted_isin(candidate_hashes, visited_hashes)
                candidate_indices = candidate_indices[keep_mask]
                candidate_hashes = candidate_hashes[keep_mask]
            if candidate_indices.numel() > 0:
                unique_pos = self._first_unique_positions(candidate_hashes)
                if unique_pos.numel() >= B or current_k == total_valid:
                    selected = unique_pos[:B]
                    keep = candidate_indices.index_select(0, selected)
                    keep_hashes = candidate_hashes.index_select(0, selected)
                    break
            if current_k == total_valid:
                empty = torch.empty((0,), dtype=torch.int64, device=self.device)
                return torch.empty((0, self.state_size), dtype=states.dtype, device=self.device), empty, empty.float(), empty, empty
            current_k = min(total_valid, current_k * 2)
        parents = torch.div(keep, self.n_gens, rounding_mode='floor')
        moves = torch.remainder(keep, self.n_gens)
        next_states = self.apply_move(states.index_select(0, parents), moves)
        return next_states, keep_hashes, q_flat[keep], moves, parents

    def normalize_moves_seq(self, moves_seq):
        if not self.normalize_path or moves_seq.numel() == 0:
            return moves_seq
        normalized = []
        pending_face = None
        pending_turns = 0
        def flush(face, turns):
            if face is None:
                return
            turns %= 5
            if turns == 1:
                normalized.append(face)
            elif turns == 2:
                normalized.extend([face, face])
            elif turns == 3:
                normalized.extend([face + "'", face + "'"])
            elif turns == 4:
                normalized.append(face + "'")
        for move_idx in moves_seq.tolist():
            name = self.move_names[int(move_idx)]
            face, delta = (name[:-1], -1) if name.endswith("'") else (name, 1)
            if pending_face is not None and face != pending_face:
                flush(pending_face, pending_turns)
                pending_turns = 0
            pending_face = face
            pending_turns += delta
        flush(pending_face, pending_turns)
        return torch.tensor([self.move_to_idx[name] for name in normalized], dtype=torch.int64) if normalized else torch.empty((0,), dtype=torch.int64)

    def get_solution(self, state, B=4096, num_steps=80, num_attempts=1):
        state = state.to(self.device)
        if torch.equal(state, self.V0):
            return torch.empty((0,), dtype=torch.int64), 0
        states_bad_hashed = torch.empty((0,), dtype=torch.int64, device=self.device)
        states = state.unsqueeze(0).clone()
        j = 0
        for attempt in range(num_attempts):
            states = state.unsqueeze(0).clone()
            tree_move = -torch.ones((num_steps, B), dtype=torch.int16)
            tree_idx = -torch.ones((num_steps, B), dtype=torch.int32)
            last_moves = -torch.ones((1,), dtype=torch.int64, device=self.device)
            visited = torch.unique(torch.cat((states_bad_hashed, state2hash(states, self.hash_vec))))
            iterator = tqdm(range(num_steps)) if tqdm is not None else range(num_steps)
            for j in iterator:
                states, state_hashes, scores, moves, parents = self.do_q_step(states, visited, last_moves, B)
                if states.size(0) == 0:
                    break
                if tqdm is not None:
                    iterator.set_description(f'q min/mean/max = {scores.min().item():.1f}/{scores.mean().item():.1f}/{scores.max().item():.1f}')
                last_moves = moves
                visited = torch.unique(torch.cat((visited, state_hashes)))
                leaves = states.size(0)
                tree_move[j, :leaves] = moves.to(dtype=tree_move.dtype, device=tree_move.device)
                tree_idx[j, :leaves] = parents.to(dtype=tree_idx.dtype, device=tree_idx.device)
                if (states == self.V0).all(dim=1).any():
                    break
            if (states == self.V0).all(dim=1).any():
                break
            states_bad_hashed = visited
        if not (states == self.V0).all(dim=1).any():
            return None, attempt
        solved_pos = torch.nonzero((states == self.V0).all(dim=1), as_tuple=True)[0].item()
        tree_idx, tree_move = tree_idx[:j + 1].flip((0,)), tree_move[:j + 1].flip((0,))
        path = [tree_idx[0, solved_pos].item()]
        for k in range(1, j + 1):
            path.append(tree_idx[k, path[-1]].item())
        moves_seq = torch.tensor([int(tree_move[k, path[k - 1]].item()) if k > 0 else int(tree_move[k, solved_pos].item()) for k in range(j + 1)], dtype=torch.int64)
        return self.normalize_moves_seq(moves_seq).flip((0,)), attempt

In [ ]:
device = torch.device(CONFIG['device'])
with paths['metadata'].open('r', encoding='utf-8') as f:
    info = json.load(f)
with paths['generator'].open('r', encoding='utf-8') as f:
    moves, move_names = parse_generator_spec(json.load(f))

all_moves = torch.tensor(moves, dtype=torch.int64, device=device)
V0 = torch_load(paths['target'], weights_only=True, map_location=device)
inverse_moves = generate_inverse_moves(move_names)
num_classes = torch.unique(V0).numel()
state_size = all_moves.size(1)
n_gens = all_moves.size(0)

model = build_model_from_info(info, num_classes=num_classes, state_size=state_size, output_dim=n_gens)
state_dict = torch_load(paths['weights'], weights_only=False, map_location='cpu')
model.load_state_dict(state_dict, strict=True)
model.eval()

if device.type == 'cuda':
    model.half()
    model.dtype = torch.float16
else:
    model.dtype = torch.float32
if V0.min() < 0:
    model.z_add = -int(V0.min().item())
model.to(device)

searcher = QSearcher(
    model=model,
    all_moves=all_moves,
    V0=V0,
    device=device,
    move_names=move_names,
    inverse_moves=inverse_moves,
    batch_size=CONFIG['eval_batch_size'],
    hash_seed=CONFIG['search_seed'],
)

print('Loaded model:', info.get('model_name'), info.get('model_id'))
print('device:', device)
print('state size:', state_size, 'num classes:', int(num_classes), 'moves:', n_gens)
print('checkpoint:', paths['weights'])

In [ ]:
# Direct Q inference: lower value means a better next move under this model.
test_state = generate_random_walk_states(
    V0=V0,
    all_moves=all_moves,
    inverse_moves=inverse_moves,
    num_states=1,
    depth=CONFIG['rnd_depth'],
    device=device,
    seed=CONFIG['rnd_seed'],
)[0]

q = searcher.pred_q(test_state.unsqueeze(0))[0].detach().float().cpu()
ranked = sorted([(move_names[i], float(q[i])) for i in range(len(move_names))], key=lambda item: item[1])
print('Top Q moves for one random-walk state:')
for name, value in ranked[:8]:
    print(f'{name:>4s}  {value:8.3f}')

In [ ]:
# Optional smoke solve. For harder states, increase beam_width and num_steps.
started = time.time()
solution, attempt = searcher.get_solution(
    test_state,
    B=CONFIG['beam_width'],
    num_steps=CONFIG['num_steps'],
    num_attempts=CONFIG['num_attempts'],
)
elapsed = time.time() - started

if solution is None:
    print(f'No solution found in {elapsed:.1f}s. Try larger beam_width/num_steps or a shallower rnd_depth.')
else:
    solution_names = [move_names[int(i)] for i in solution.tolist()]
    print(f'Solved in {len(solution_names)} moves, attempt={attempt + 1}, time={elapsed:.1f}s')
    print(' '.join(solution_names))

## Switching Checkpoints

To run another Q checkpoint, edit `CONFIG['model_name']`, `CONFIG['model_id']`, and `CONFIG['checkpoint']`, then make sure matching files exist under `logs/` and `weights/`.

For the older public release checkpoint, use:

```python
CONFIG['model_name'] = 'p900-t000-q'
CONFIG['model_id'] = 1776581286
CONFIG['checkpoint'] = 'best'
```

The notebook code is self-contained; only the heavy model artifacts stay external.